In [1]:
from typing import Optional
from core.db import engine
from sqlalchemy import select
from adapters.linkedin.models import JobObservation

def get_job_observations(engine, limit: Optional[int] = None):
    stmt = select(JobObservation)
    if limit:
            stmt = stmt.limit(limit)
    with engine.connect() as conn:
        return [dict(row) for row in conn.execute(stmt).mappings()]

def get_job_observations_by_ids(engine, ids: list[str]):
    stmt = select(JobObservation).where(JobObservation.job_id.in_(ids))
    with engine.connect() as conn:
        return [dict(row) for row in conn.execute(stmt).mappings()]


In [2]:
jobs = get_job_observations(engine, limit= 1)
sample_desc = jobs[0]['description']
len(sample_desc)

3203

In [ ]:
from enrichment.linkedin.llm import JobAdParser
job_ad_parser = JobAdParser(model_name="gpt-4o-mini") # nvidia/nemotron-3-ultra-550b-a55b:free

In [ ]:
COUNT = 20
results = {}
jobs = get_job_observations(engine, limit= COUNT)
for job in jobs:
    job_id = job["job_id"]
    results[job_id] = job_ad_parser.parse(job['description'])

In [27]:
def show(profile):
    lines = []
    if profile.global_years_of_experience is not None:
        lines.append(f"Years: {profile.global_years_of_experience}")
    if profile.programming_languages:
        langs = ", ".join(f"{p.name} ({p.experience})" for p in profile.programming_languages)
        lines.append(f"Languages: {langs}")
    if profile.tech_stack:
        lines.append("Tech: " + ", ".join(t.name for t in profile.tech_stack))
    if profile.cloud_services:
        clouds = ", ".join(f"{c.cloud_provider} {c.cloud_service}".strip() for c in profile.cloud_services)
        lines.append(f"Cloud: {clouds}")
    if profile.technical_skills:
        lines.append("Skills: " + ", ".join(s.name for s in profile.technical_skills))
    if profile.soft_skills:
        lines.append("Soft: " + ", ".join(s.desc for s in profile.soft_skills))
    if profile.domain_knowledge:
        lines.append("Domain: " + ", ".join(profile.domain_knowledge))
    if profile.certifications:
        lines.append("Certs: " + ", ".join(profile.certifications))
    if profile.education:
        lines.append(f"Education: {profile.education}")
    print("\n".join(lines) or "(empty)")

In [29]:
jobs = get_job_observations(engine, limit= COUNT)

In [31]:
for job, res in zip(jobs, results): 
    print(job["job_id"], job["title"])
    show(results[res])
    print('-' * 40)

4426630888 Data Engineer (Financial Analytics focus)
Years: 4
Languages: Python (Intermediate)
Tech: PostgreSQL, Airflow, GitLab, GitHub, JIRA
Skills: Data pipeline development, Data transformation, Data quality checks, Data warehouse architecture, ELT/ETL pipeline construction, Automated data quality checks
Soft: Stakeholder collaboration, Autonomy
Domain: financial, trading
----------------------------------------
4435407777 Data Engineer
Tech: Azure Data Warehouse, Azure Data Factory, Power Platform, Synapse, Databricks, Purview, Power BI
Cloud: Azure Data Factory
Skills: Data modelling, CI/CD, Agile methodologies, Data governance, Performance optimization
Soft: Problem solving, Team collaboration
Domain: Water management
Education: Bachelors in Computer Science
----------------------------------------
4400493686 AI Engineer
Years: 2
Languages: Python (2 Years)
Tech: Langflow, Terraform, Kubernetes, Docker
Cloud: AWS
Skills: AI solution development, Solution lifecycle management, In

In [44]:
jobs = get_job_observations_by_ids(engine, ["4433923261","4439839314"])

In [46]:
results2 = {}
for job in jobs:
    job_id = job["job_id"]
    results2[job_id] = job_ad_parser.parse(job['description'])

In [47]:
for job, res in zip(jobs, results2): 
    print(job["job_id"], job["title"])
    show(results2[res])
    print('-' * 40)

4433923261 Junior Machine Learning Engineer
Years: 0
Languages: Python (Familiar)
Tech: PyTorch, OpenCV, scikit-learn, scikit-image, PIL, Pandas
Skills: ETL process, Deep learning, Computer vision, Data pipelines
Soft: Curiosity, Collaboration
----------------------------------------
4439839314 Data Engineer - Data Platforms
Tech: Snowflake, dbt, Prefect, Terraform, Flink
Cloud: AWS AWS
Skills: Data modeling, Pipeline design, Data quality management, CI/CD, Infrastructure as Code
Soft: Problem-solving, Collaboration, Ownership
----------------------------------------


In [48]:
jobs[0]["description"]

"\n## \n## Who \n## \n## we are\n\nArkeus builds AI-powered sensing systems that help autonomous platforms detect, track and understand what's happening in the most complex and contested environments. Our technology is deployed across defence and national security applications — already protecting lives and strengthening security across the U.S. and allied markets.\n\nBacked by QIC, Main Sequence, R+VC, Salus Ventures, Folklore Ventures, DYNE Ventures and Beaten Zone Venture Partners, Arkeus is entering a major growth phase. We are a small, fast-moving company with the energy and ambition of a start-up and the technical credibility of a world-class defence technology provider.\n\nWe work at the intersection of optics, autonomy, robotics and real-time AI and we deploy what we build. If you want to ship technology that matters, not technology that ships decks, this is where you belong.\n\nCheck it yourself on News | Arkeus\n\nWe have developed an inclusive, transparent and engaging cultu